# Lab 00: How to check your lab environment

**CPU | API key: `OPENAI_API_KEY` (three short requests in Parts F to H)**

## Goal

Check in one run that your setup can run every lab in this repository, and see which models and endpoint the labs
will use. Set up first with this lab's README (your own machine with `uv` or `pip`, GitHub Codespaces, an Ubuntu VM
or Colab), then run every cell from the top. When the last cell shows ✅ on every line, you are ready for Lab 01.

## What you will learn

- What the first two cells of every lab do, on every route.
- Where the model names, the API address and your key come from, and which value wins when two places set it.
- How to read the stamp every result in these labs carries: which models, which endpoint, which day.
- How to tell a blocked network from a key problem.
- How to find what a failed check means and fix it with the troubleshooting table.

## The first two cells

Every lab in this repository opens with the same two cells.

**Setup** finds the repository folder. On Colab it first copies the repository into the session (`git clone`) and
installs the packages in `requirements.txt`. On your own machine, in a codespace or on a VM, you installed the
packages once with the README's steps, so the cell installs nothing.

**Config** reads the settings and looks for your key. It prints the settings that decide every result, and where it
found the key, never the key itself.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


The last three lines find the repository folder: `next(...)` looks in the folder the notebook runs in, then in each
folder above it, and takes the first one that holds `models.env`. Adding that folder to `sys.path` lets the next cell
import from `shared/`, where the labs keep the code they reuse.

In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-18
OPENAI_API_KEY    found in .env


Each setting can come from three places, and a later one wins:

1. `models.env` in the repository folder: the shipped values, the same for everyone in the class.
2. Your `.env` file, when it sets the same name: your machine only.
3. An environment variable with the same name, set by your system, a codespace or a terminal.

To switch every lab to another model, change one line in `models.env`; to try one on your machine only, put the line
in your `.env`. No notebook names a model in its code.

The lines above the key are the **stamp**. `stamp()` in `shared/settings.py` returns them, and every result a lab
saves and every table it prints carries them, so when two people compare numbers they can see whether the same
models on the same endpoint produced them. `reasoning_effort` is `None` while `REASONING_EFFORT` is empty, as it
is for `gpt-4o-mini` (Part G). The key is looked for in the environment first, where a codespace puts its secrets, then in `.env`, then in
Colab Secrets.

If the key line says `not found`, Parts A to E still run; add the key before Part F (troubleshooting table at the
end). The outputs saved here come from a run on macOS with `uv` (Python 3.13.5) on 18 October 2026, with `gpt-4o-mini` and `text-embedding-3-small`.

---

## Part A: Check Python and the packages

The saved outputs in every lab came from the package versions in `uv.lock`. `requirements.txt` lists the same
packages with the range of versions that work, for `pip` and Colab. This cell reads each line, asks the environment
the notebook runs in which version is installed (`version(name)`), and checks that it falls inside the range. That
environment is the one that matters: a terminal can use a different Python.

In [3]:
import platform
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement

outside = []
for line in (ROOT / "requirements.txt").read_text().splitlines():
    if not line or line.startswith("#"):
        continue
    requirement = Requirement(line)                # a package name and its range, such as openai>=2.54,<4
    try:
        installed = version(requirement.name)
    except PackageNotFoundError:
        installed = "missing"
    inside = installed != "missing" and requirement.specifier.contains(installed, prereleases=True)
    if not inside:
        outside.append(requirement.name)
    print(f"{requirement.name:26}{installed:10}{str(requirement.specifier):12}{'' if inside else 'OUTSIDE'}")

python_ok = (3, 11) <= sys.version_info[:2] <= (3, 14)
note = "" if python_ok else " (the labs need 3.11 to 3.14)"
print(f"\nPython {platform.python_version()}{note} | outside their range: {outside or 'none'}")

openai                    3.35.1    <4,>=2.54   
numpy                     2.5.4     <3,>=2.0    
tiktoken                  0.14.0    <1,>=0.8    
python-dotenv             1.2.4     <2,>=1.0    
bm25s                     0.3.13    <0.4,>=0.3  
langchain                 1.4.6     <2,>=1.4    
langchain-openai          1.7.2     <2,>=1.7    
langchain-text-splitters  1.1.3     <2,>=1.1    
pytest                    9.1.1     <10,>=8     

Python 3.13.5 | outside their range: none


A pass is Python 3.11 to 3.14 and `outside their range: none`. Ours: Python 3.13.5 with the versions in `uv.lock`, among them `openai` 3.35.1 and `langchain` 1.4.6. A `pip` install can show newer versions
inside each range; that is fine. A package that is `missing` or `OUTSIDE` means the notebook runs on another kernel
or the install did not finish (troubleshooting table).

---

## Part B: Check the data files

Every lab reads the same frozen data from `data/`: 120 Wikipedia paragraphs, the same paragraphs split into 272
chunks, 30 development questions and five short product policies. Many checkpoints depend on the exact text.
`data/SHA256SUMS.txt` lists a SHA-256 hash of each file, a 64-character fingerprint of its bytes that changes
completely when one byte changes. The cell hashes each file and compares.

In [4]:
import hashlib

DATA = ROOT / "data"
sums = (DATA / "SHA256SUMS.txt").read_text().splitlines()
changed, missing = [], []
for line in sums:
    expected, name = line.split("  ")              # the hash, two spaces, the file name
    path = DATA / name
    if not path.exists():
        missing.append(name)
    elif hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        changed.append(name)

print(f"{len(sums) - len(changed) - len(missing)} of {len(sums)} files match")
print("Changed:", changed or "none")
print("Missing:", missing or "none")

5 of 5 files match
Changed: none
Missing: none


A pass is 5 of 5 files, none changed and none missing. Saving a data file from an editor can be enough to change its
hash, which is why the labs only read `data/`.

---

## Part C: Check the settings

Config printed the settings that go into the stamp. This cell lists every setting in `models.env` and the place its
value came from, so a line you forgot in `.env` shows up here instead of as an odd result three labs later.

In [5]:
import os
from dotenv import dotenv_values

local = dotenv_values(ROOT / ".env")
for name in settings:
    source = "environment" if os.environ.get(name) else ".env" if local.get(name) else "models.env"
    print(f"{name:24}{settings[name] or '(empty)':26} from {source}")

setting_problems = [name + " is empty" for name in ("CHAT_MODEL", "EMBED_MODEL", "JUDGE_MODEL", "RERANK_MODEL")
                    if not settings[name]]
if settings["JSON_MODE"] not in ("json_schema", "json_object"):
    setting_problems.append("JSON_MODE must be json_schema or json_object")
print("\nProblems:", setting_problems or "none")

OPENAI_BASE_URL         (empty)                    from models.env
EMBED_BASE_URL          (empty)                    from models.env
CHAT_MODEL              gpt-4o-mini                from models.env
EMBED_MODEL             text-embedding-3-small     from models.env
JUDGE_MODEL             gpt-4o-mini                from models.env
RERANK_MODEL            gpt-4o-mini                from models.env
REASONING_EFFORT        (empty)                    from models.env
JSON_MODE               json_schema                from models.env
TOKEN_ENCODING          o200k_base                 from models.env
EMBED_DIMENSIONS        (empty)                    from models.env
CHAT_PRICE_INPUT        0.15                       from models.env
CHAT_PRICE_CACHED_INPUT 0.075                      from models.env
CHAT_PRICE_OUTPUT       0.60                       from models.env
EMBED_PRICE_INPUT       0.02                       from models.env

Problems: none


A pass is `Problems: none`: every model is named and `JSON_MODE` holds one of its two values. In our run every value came from `models.env`. The comment
above each line in `models.env` says what it does; the labs that use a setting explain it there.

---

## Part D: Check that you can reach the endpoint

This part tests the network alone. It sends one request without your key to each endpoint's `/models` address. Any
HTTP answer, even `401` (no key sent), means your network lets the labs through. No answer means a firewall or proxy
in the way, a wrong address, or no connection. Part F then tests the key, so the two problems never look alike: the
network is a question for IT, the key for your instructor or your account page.

In [6]:
import urllib.error
import urllib.request
from shared.settings import chat_url, embed_url

endpoint_ok = True
for url in dict.fromkeys([chat_url(settings), embed_url(settings)]):   # each address once
    try:
        urllib.request.urlopen(url + "/models", timeout=15)
        print(url, "answered: HTTP 200")
    except urllib.error.HTTPError as error:        # the server answered, with an error status
        print(url, "answered: HTTP", error.code)
    except OSError as error:                       # no answer at all
        endpoint_ok = False
        print(url, "no answer:", error)

https://api.openai.com/v1 answered: HTTP 401


A pass is an answer from each endpoint. Ours: `https://api.openai.com/v1 answered: HTTP 401`. The troubleshooting table lists the usual "no answer"
messages, such as `CERTIFICATE_VERIFY_FAILED` from a company proxy that inspects web traffic.

---

## Part E: Check the tokenizer

Some labs count a prompt's tokens before they send it, with `tiktoken`. The first time it uses an encoding,
`tiktoken` downloads it from `openaipublic.blob.core.windows.net` and keeps a copy on disk, so this is a second
network check, on a different address. From here on each cell catches its own error and prints it, so one failure
does not stop the cells after it.

In [7]:
import tiktoken

question = "In what year was Nikola Tesla born?"           # one of the 30 development questions
try:
    encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])
    tokens = encoding.encode(question)
    print(f"{settings['TOKEN_ENCODING']}: {len(tokens)} tokens {tokens}")
    tokenizer_ok = True
except Exception as error:
    tokenizer_ok = False
    print("Failed:", type(error).__name__, "-", error)

o200k_base: 8 tokens [637, 1412, 1284, 673, 136985, 52960, 12275, 30]


A pass is a token count. Ours: 8 tokens for the question with `o200k_base`. Lab 03 explains what the numbers are.

---

## Part F: Check the key with one embedding

Now the key. This cell sends the same question to the embedding model, which returns a list of numbers that stands
for the question's meaning; Lab 01 is built on these. `embed_client(settings)` makes an OpenAI client for the
embedding endpoint in the settings. Lab 01 makes the same client by hand and shows this function's code.

⚠️ Parts F to H send three short requests with your key. Together they cost a small fraction of a US cent.

In [8]:
import math
from shared.settings import embed_client

try:
    response = embed_client(settings).embeddings.create(model=settings["EMBED_MODEL"], input=question)
    vector = response.data[0].embedding
    print(f"{len(vector)} numbers, starting {[round(x, 4) for x in vector[:3]]}")
    print(f"length {math.sqrt(sum(x * x for x in vector)):.3f} | {response.usage.total_tokens} tokens billed")
    embed_ok = len(vector) > 0
except Exception as error:
    embed_ok = False
    print("Failed:", type(error).__name__, "-", error)

1536 numbers, starting [-0.032, -0.0686, -0.0369]
length 1.000 | 8 tokens billed


A pass is a list of numbers. With `text-embedding-3-small` it holds 1536 numbers and its length is 1.000; Lab 01
explains why the length matters. Ours: 1536 numbers, length 1.000, 8 tokens billed. An error here after a pass in Part D is a key or account
problem, not the network.

---

## Part G: Check the chat model

One question to each chat model in the settings: `CHAT_MODEL`, and `JUDGE_MODEL` and `RERANK_MODEL` when they name
a different model. `chat_options()` adds the model name and, depending on `REASONING_EFFORT`, `temperature` or
`reasoning_effort`; the first line prints what it adds. Lab 03 shows how it decides.

In [9]:
from shared.settings import chat_client, chat_options

models = {}                                        # one call per distinct model
for name in ("CHAT_MODEL", "JUDGE_MODEL", "RERANK_MODEL"):
    models.setdefault(settings[name], name)

chat_ok = True
for model, name in models.items():
    options = chat_options(name, settings)
    print(name, "sends", options)
    try:
        reply = chat_client(settings).chat.completions.create(
            **options, messages=[{"role": "user", "content": question + " Answer with the year only."}])
        print("  reply:", reply.choices[0].message.content, "| answered by", reply.model)
        print("  usage:", reply.usage.prompt_tokens, "tokens in,", reply.usage.completion_tokens, "out")
    except Exception as error:
        chat_ok = False
        print("  Failed:", type(error).__name__, "-", error)

CHAT_MODEL sends {'model': 'gpt-4o-mini', 'temperature': 0}


  reply: 1856 | answered by gpt-4o-mini-2024-07-18
  usage: 21 tokens in, 2 out


A pass is a reply in any wording. Ours: `1856`, answered by `gpt-4o-mini-2024-07-18`, 21 tokens in and 2 out. `reply.model` names the model that answered: `gpt-4o-mini` in
`models.env` is an alias, and OpenAI reports the dated version behind it. `usage` is what the endpoint billed.

`gpt-4o-mini` is due to shut down on 14 January 2027, and its replacement, `gpt-6-luna`, is a reasoning model. To
switch, set `CHAT_MODEL`, `JUDGE_MODEL` and `RERANK_MODEL` to `gpt-6-luna` and `REASONING_EFFORT=none` together.
`REASONING_EFFORT` goes to every chat model and `gpt-4o-mini` rejects it; with `none`, `gpt-6-luna` still accepts
`temperature`, and with the setting empty it rejects `temperature: 0`. We ran this notebook on `gpt-6-luna` as well: Part G sent `{'model': 'gpt-6-luna', 'reasoning_effort': 'none', 'temperature': 0}`, the reply was `1856` (20 tokens in, 5 out), and every check passed.

---

## Part H: Check JSON replies

Several labs ask the model for JSON that follows a schema and then check every reply in code. `JSON_MODE` in the
settings says how they ask: `json_schema` makes the endpoint follow a strict schema; `json_object` only promises
valid JSON, for servers without strict schemas. This cell asks for `{"year": <a number>}` the way the settings say
and checks that the reply has that shape.

In [10]:
import json

schema = {"type": "object", "properties": {"year": {"type": "integer"}},
          "required": ["year"], "additionalProperties": False}
if settings["JSON_MODE"] == "json_schema":
    response_format = {"type": "json_schema", "json_schema": {"name": "year", "strict": True, "schema": schema}}
else:
    response_format = {"type": "json_object"}

try:
    reply = chat_client(settings).chat.completions.create(
        **chat_options("CHAT_MODEL", settings), response_format=response_format,
        messages=[{"role": "user", "content": question + ' Reply in JSON: {"year": <the year as a number>}'}])
    parsed = json.loads(reply.choices[0].message.content)
    json_ok = list(parsed) == ["year"] and isinstance(parsed["year"], int)
    print(settings["JSON_MODE"], "->", parsed, "| right shape" if json_ok else "| wrong shape")
except Exception as error:
    json_ok = False
    print("Failed:", type(error).__name__, "-", error)

json_schema -> {'year': 1856} | right shape


A pass is a reply with one key, `year`, holding a whole number. Ours: `json_schema -> {'year': 1856} | right shape`. If Part G passes and this part
fails with `json_schema`, your endpoint does not support strict schemas: set `JSON_MODE=json_object` in your `.env`,
then run Config and this part again.

---

## Check that you are ready

One line per check. If a line fails, copy the whole output when you ask for help: the first two lines say where the
notebook ran and with which models.

In [11]:
from datetime import date

checks = {
    "Config  OPENAI_API_KEY found": bool(key_found_in),
    "A  Python and the packages": python_ok and not outside,
    "B  data files unchanged": not changed and not missing,
    "C  settings complete": not setting_problems,
    "D  endpoint reachable": endpoint_ok,
    "E  tokenizer loads": tokenizer_ok,
    "F  embedding call works": embed_ok,
    "G  chat model answers": chat_ok,
    "H  JSON replies have the right shape": json_ok,
}
where = "Colab" if IN_COLAB else "Codespaces" if os.environ.get("CODESPACES") == "true" else platform.platform(terse=True)
print("Run on", date.today().isoformat(), "|", where, "| Python", platform.python_version())
print("Models:", settings["CHAT_MODEL"], "and", settings["EMBED_MODEL"], "| endpoint:", chat_url(settings))
for name, passed in checks.items():
    print("✅" if passed else "❌", name)

Run on 2026-10-18 | macOS-15.7.9 | Python 3.13.5
Models: gpt-4o-mini and text-embedding-3-small | endpoint: https://api.openai.com/v1
✅ Config  OPENAI_API_KEY found
✅ A  Python and the packages
✅ B  data files unchanged
✅ C  settings complete
✅ D  endpoint reachable
✅ E  tokenizer loads
✅ F  embedding call works
✅ G  chat model answers
✅ H  JSON replies have the right shape


Every line passes: you are ready for Lab 01. A ❌ names the check to look at; the troubleshooting table below lists
the usual causes.

### Checkpoint

| Check | You should see | Our run |
|---|---|---|
| Config | the stamp, and `OPENAI_API_KEY found in ...` | the stamp above; key found in `.env` |
| A | Python 3.11 to 3.14; outside their range: none | Python 3.13.5; outside their range: none |
| B | 5 of 5 files match; changed: none; missing: none | 5 of 5; none changed; none missing |
| C | Problems: none | Problems: none; every value from `models.env` |
| D | an HTTP answer from each endpoint | `https://api.openai.com/v1 answered: HTTP 401` |
| E | a token count | 8 tokens |
| F | 1536 numbers, length 1.000 | 1536 numbers, length 1.000 |
| G | a reply in any wording | `1856`, from `gpt-4o-mini-2024-07-18` |
| H | `{'year': ...}`, right shape | `{'year': 1856}`, right shape |

## What you can do now

- Open any lab in this repository on your route; its first two cells are the ones you ran here.
- Find the line that chooses a model, and change it for every lab (`models.env`) or on your machine only (`.env`).
- Read a stamp and say which models and endpoint produced a result.
- Tell from Parts D and F whether a failure is the network or the key.
- Find what a failed check means in the troubleshooting table and fix it.

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| Setup stops with `StopIteration` | the notebook is not inside the repository folder | open it from the folder you cloned or unzipped, where `models.env` sits |
| Config stops with `ModuleNotFoundError: No module named 'shared'` | Setup did not run first | run Setup, then Config |
| Config stops with `ModuleNotFoundError: No module named 'dotenv'`, and Part A lists packages as `missing` | the notebook runs on a kernel without the packages | choose the `.venv` kernel (README, step 5 of your route) |
| Part A says `(the labs need 3.11 to 3.14)` | an older or newer Python; Ubuntu 22.04's own Python is 3.10 | use `uv` (README, route A), which installs Python 3.13 for the repository |
| Part A marks a package `OUTSIDE` its range | an older or newer copy in the environment | with `uv`: `uv sync`; with `pip`: `pip install -r requirements.txt` in the activated environment. Then restart the kernel and run from the top |
| Part B lists a file as changed or missing | the file was edited, saved by an editor or deleted | `git checkout -- data/<file name>`, or download the file again from the repository |
| Part C shows `Problems: [...]`, or a value `from .env` you did not expect | a line in your `.env` overrides `models.env` | correct or delete that line in `.env`, then run Config and Part C again |
| Config says the key is `not found`, and Parts F to H print `OpenAIError - Missing credentials` | no key in the environment, `.env` or Colab Secrets | add it (README, step 4 of your route; Codespaces: add the secret and restart the codespace), then run Config again |
| Part D: `no answer: <urlopen error [Errno 61] Connection refused>` or `nodename nor servname provided` (Linux: `[Errno 111]` or `Name or service not known`); Parts F to H: `APIConnectionError - Connection error.` | a wrong `OPENAI_BASE_URL`, no connection, or a firewall | check `OPENAI_BASE_URL` in Part C (empty means OpenAI); try another network; ask IT to allow the address |
| Part D: `no answer: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] ...>` | a company proxy that inspects web traffic | use another network, a codespace or Colab, or ask IT how Python should trust the company's certificate |
| Part E: `ProxyError` or `ConnectionError` for `openaipublic.blob.core.windows.net` | the network blocks the tokenizer's one-time download | ask IT to allow the address, or copy the cache folder from a machine where Part E passed and set the environment variable `TIKTOKEN_CACHE_DIR` to it |
| Parts F to H: `AuthenticationError - Error code: 401` with `Incorrect API key provided` or `Your API key has expired` | the key is mistyped, revoked or expired | paste the key again (no quotes, no spaces) or create a new one; for a class key, ask the instructor |
| Part G: `NotFoundError - Error code: 404` with ``The model `...` does not exist or you do not have access to it`` | a misspelt model name, or your account cannot use that model | check the model lines in Part C and in your `.env` |
| Part G: `BadRequestError - Error code: 400` with `Unrecognized request argument supplied: reasoning_effort` | `REASONING_EFFORT` is set, and one of the chat models (often `JUDGE_MODEL` or `RERANK_MODEL`) has no such setting | switch all three chat models to the reasoning model, or empty `REASONING_EFFORT` |
| Part G: `BadRequestError - Error code: 400` with `Unsupported value: 'temperature' does not support 0 with this model` | a reasoning model such as `gpt-6-luna` with `REASONING_EFFORT` empty | set `REASONING_EFFORT=none` |
| Parts F to H: `RateLimitError - Error code: 429` | a shared class key reached its limit, or the account has no credit | wait a minute and run the cell again; tell the instructor if it repeats |
| Part H fails while Part G passes | the endpoint does not support strict JSON schemas | set `JSON_MODE=json_object` in your `.env`, then run Config and Part H again |

The README has a second table for problems during setup, before the notebook opens.

## Resources

- [OpenAI: embeddings](https://developers.openai.com/api/docs/guides/embeddings): the call in Part F and the
  embedding models you can set in `models.env`
- [OpenAI: Chat Completions API reference](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create):
  the call in Parts G and H, with `temperature`, `reasoning_effort` and `response_format`
- [OpenAI: structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs): `json_schema`
  against `json_object`, the choice in Part H
- [OpenAI: deprecations](https://developers.openai.com/api/docs/deprecations): shutdown dates for models such as
  `gpt-4o-mini`
- [tiktoken](https://github.com/openai/tiktoken): the tokenizer in Part E
- [Python `hashlib`](https://docs.python.org/3/library/hashlib.html): the SHA-256 hashes in Part B

Next: [Lab 01: How to search documents with embeddings](../01_Search_With_Embeddings/README.md). Back to
[the lab list](../README.md).